# Lesson 12 | The complete journey of one spike

The last three lessons covered shared compute, spike queues, and sparse source lookup. Today connects them:
> **How does a source spike leave a queue, become target updates, and possibly create new spikes?**

Primary concept: the end-to-end causal chain of **event-driven computation**.


## 1. Concept ledger

**Known:** time multiplexing, FIFO/backpressure, and source_index plus sparse synapse records.

**New:** composing them into an event-driven pipeline; supporting terms include router, weighted event, and synapse stream.

**Preview:** formal valid/ready streaming, concurrent write conflicts, and high-performance target accumulation come later.


## 2. What does event-driven mean?

Instead of scanning every possible connection every cycle, the system processes a source's real downstream synapses **only when a spike event occurs**.

This changes how computation is organized; it does not redefine the biological neuron model.


## 3. One complete path

```mermaid
flowchart LR
 Q["spike FIFO"] --> S["source_id"]
 S --> IDX["source index lookup"]
 IDX --> R["synapse records"]
 R --> W["weighted events: target, weight"]
 W --> A["target accumulator / update"]
 A --> T["threshold rule"]
 T -->|new spike| Q
```

A **router** is the control/lookup logic that decides where a source event goes next. A **synapse stream** is the ordered stream of `(source,target,weight,...)` records produced by lookup.


## 4. Four-neuron hand-written network

To isolate the event journey, this lesson uses a tiny teaching threshold accumulator rather than formal LIF:

- 0 → 1 with weight +2
- 0 → 2 with weight +1
- 1 → 3 with weight +2
- 2 → 3 with weight +1

Thresholds are neuron 1=2, 2=1, and 3=3. Initially only neuron 0 spikes.

Predict the spike order before running.


In [ ]:
from collections import deque

# Four-neuron teaching network.
# records for source 0: (1,+2), (2,+1)
# source 1: (3,+2), source 2: (3,+1), source 3: none
source_index = [(0, 2), (2, 1), (3, 1), (4, 0)]
records = [(1, 2), (2, 1), (3, 2), (3, 1)]
threshold = [99, 2, 1, 3]
accum = [0, 0, 0, 0]
queue = deque([0])
spike_order = []

while queue:
    source = queue.popleft()
    spike_order.append(source)
    print(f'\nconsume spike source={source}')
    start, count = source_index[source]
    for target, weight in records[start:start+count]:
        accum[target] += weight
        print(f'  weighted event -> target={target}, weight={weight}, accum={accum[target]}')
        if accum[target] >= threshold[target]:
            print(f'  target {target} spikes; enqueue it and reset teaching accumulator')
            accum[target] = 0
            queue.append(target)

print('\nspike order:', spike_order)
print('final accum:', accum)


## 5. Observe the causal chain, not only the final value

Trace it step by step: source 0 leaves the queue; lookup produces two real synapses; targets 1 and 2 reach threshold and enter the queue; each later contributes to target 3; target 3 reaches 3 and emits a new spike.

Expected spike order: `[0, 1, 2, 3]`.


## 6. A crucial boundary

This Python code is a **teaching event machine**, not the formal neuron model. It deliberately omits leak, fixed point, refractory behavior, concurrent target-write conflicts, and valid/ready timing.

Do not copy its numerical semantics into formal `MOD-003`. This lesson verifies routing causality only.


## 7. Try It: remove one edge

Predict first: if `2 → 3, +1` is removed, will neuron 3 still spike? Why?

Then remove that record and update source_index consistently. The experiment also shows why sparse indexes and record arrays must stay synchronized.


## 8. Homework

Complete `process_one_spike(...)` in `exercises/lesson12_event_journey.py`: given one source event, process only its synapse range and return weighted events plus updated target accumulators.

```bash
uv run pytest exercises/checks/check_lesson12.py -q
```


## 9. AI Task

Give AI a failing event trace and require it to locate the first causal mismatch across `queue → source lookup → synapse record → target accumulation`. It may not start by changing the neuron threshold.


## 10. Human Check

Without AI, derive `[0,1,2,3]` from source 0, state what the queue stores, what source_index selects, what a synapse record carries, what target accumulation changes, and why event-driven computation does not mean neuron state exists only when spikes occur.


## 11. Engineering Handoff

This lesson implements the RMD-007A four-neuron walk-through conceptually and prepares RMD-010 synapse reader + engine and RMD-011 small event-driven SNN. Formal `MOD-005~009` and T-010~T-013 still require module-by-module implementation and verification.


## 12. Project Trace

- Lesson: `LSN-012`
- Mapping: `RMD-007A / RMD-010 / RMD-011` teaching integration
- Module context: `MOD-005~009`
- Test context: prepares `T-010~T-013`


## 13. Exit Ticket

Without code, you can explain the full path from source event to target update to a newly enqueued spike, and identify which kind of test should verify each stage.
